## Artisitc Portrait Gen

In [ ]:
import torch
from artistic_portrait.pipeline import ArtisticPortraitXLPipeline
from diffusers import ControlNetModel
from PIL import Image
from ip_adapter_diffusers.ip_adapter import *
from diffusers import DPMSolverMultistepScheduler
from huggingface_hub import hf_hub_download
import os

In [ ]:
device = "cuda"
dtype = torch.float16
style_adapter_path = "models/ip_adapter_art_sdxl_512.pth"
id_adapter_path = "models/pulid_adapter_diffusers_1.1.pth"

In [ ]:
if not os.path.exists("models/csd_clip.pth"):
    hf_hub_download(
        repo_id="AisingioroHao0/IP-Adapter-Art",
        filename="csd_clip.pth",
        local_dir="models",
    )
if not os.path.exists(style_adapter_path):
    hf_hub_download(
        repo_id="AisingioroHao0/IP-Adapter-Art",
        filename="ip_adapter_art_sdxl_512.pth",
        local_dir="models",
    )
if not os.path.exists(id_adapter_path):
    hf_hub_download(
        repo_id="AisingioroHao0/IP-Adapter-Art",
        filename="pulid_adapter_diffusers_1.1.pth",
        local_dir="models",
    )

In [ ]:
controlnet = ControlNetModel.from_pretrained(
    "xinsir/controlnet-openpose-sdxl-1.0",
    torch_dtype=dtype,
).to(device)
pipe = ArtisticPortraitXLPipeline.from_pretrained(
    "stabilityai/stable-diffusion-xl-base-1.0",
    controlnet=controlnet,
    safety_checker=None,
    torch_dtype=torch.float16,
    style_adapter_path=style_adapter_path,
    id_adapter_path=id_adapter_path,
    variant="fp16",
    device=device,
).to(device)
pipe.scheduler = DPMSolverMultistepScheduler.from_config(
    pipe.scheduler.config, timestep_spacing="trailing"
)

In [ ]:
height = 1024
width = 1024
artify_controlnet_scale = 0.0
style_scale = 1.0
id_scale = 1.0
controlnet_scale = 0.9

if artify_controlnet_scale > 0:
    pipe.load_style_adapter_to_controlnet(style_adapter_path)
    set_ip_adapter_scale(pipe.controlnet, artify_controlnet_scale)

style_image = Image.open("datasets/test/style_dataset/Abstract D'Oyley.jpg")
id_image = Image.open("datasets/test/id_dataset/hinton.jpg")
pose_image = Image.open("datasets/test/pose.jpg")

In [ ]:
result = pipe(
    f"portrait, solo, looking at viewer, best quality, masterpiece",
    negative_prompt="flaws in the eyes, flaws in the face, flaws, lowres, non-HDRi, low quality, worst quality,artifacts noise, text, watermark, glitch, deformed, mutated, ugly, disfigured, hands, low resolution, partially rendered objects,  deformed or partially rendered eyes, deformed, deformed eyeballs, cross-eyed",
    control_image=pose_image,
    controlnet_conditioning_scale=controlnet_scale,
    width=width,
    height=height,
    num_inference_steps=20,
    guidance_scale=7,
    style_image=style_image,
    id_image=id_image,
    generator=torch.Generator("cuda").manual_seed(42),
    id_scale=1.0,
    style_scale=1.0,
    # num_zero=[None, 16],
    # ortho=[None, 'ortho_v2'],
).images[0]
result